In [46]:
import torch
import numpy as np

from sklearn.datasets import load_wine
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, CrossEntropyLoss, Module
from torch.utils.data import DataLoader, Dataset
from torch.optim import Adam
from sklearn.preprocessing import StandardScaler


In [47]:
class MyDataset(Dataset):
    def __init__(self, datas: np.ndarray, labels: np.ndarray) -> None:
        self._x = torch.from_numpy(datas).float()
        self._y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return len(self._x)

    def __getitem__(self, idx: int) -> tuple[Tensor, Tensor]:
        int(idx)
        return self._x[idx], self._y[idx]

In [48]:
class CustomModel(Module):
    def __init__(self, input_size: int, output_size) -> None:
        super().__init__()
        self.net = Sequential(
            Linear(input_size, 32),
            ReLU(),
            Linear(32,16),
            ReLU(),
            Linear(16, output_size)
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)

In [49]:
def train_model_reg(X, Y, skf, batch, epochs, criterion):
    f1_scores = []
    accuracy_scores = []
    for i, (train_index, test_index) in enumerate(skf.split(X, Y)):
        print(f"Treinando modelo ----- Fold {i + 1}")
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X[train_index])
        X_test_scaled = scaler.transform(X[test_index])

        train_data = MyDataset(X_train_scaled, Y[train_index])
        train_loader = DataLoader(train_data, batch_size=batch)
        model = CustomModel(13, 3)
        optmizer = Adam(model.parameters(), lr=0.001)

    
        for i_epochs in range(epochs):    
            model.train()
            for x, y in train_loader:
                optmizer.zero_grad()
                output = model(x).cpu()
                loss = criterion(output, y)
                loss.backward()
                optmizer.step()

        model.eval()
        with torch.no_grad():
            x_test_tensor = Tensor(X_test_scaled).float()
            output = model(x_test_tensor).cpu()
            result = torch.argmax(output, dim=1).cpu().numpy()
            f1_s = f1_score(Y[test_index], result, average="macro")
            accuracy_s = accuracy_score(Y[test_index], result)
            f1_scores.append(f1_s)
            accuracy_scores.append(accuracy_s)
            print(f"f1: {f1_s}")
            print(f"Accuracy: {accuracy_s}")

    print(f"\nResultado modelo ----- F1: {np.mean(f1_scores)},  accuracy:{np.mean(accuracy_scores)}")

In [50]:
dataset = load_wine()
X = dataset.data
Y = dataset.target

In [51]:
print(X.shape)

(178, 13)


In [52]:
model = Sequential(
    Linear(13, 32),
    ReLU(),
    Linear(32,16),
    ReLU(),
    Linear(16, 3)
)

In [53]:

criterion = CrossEntropyLoss()
max_epochs = 150
batch = 16
skf = StratifiedKFold(n_splits=5, shuffle=True,  random_state=42)

In [54]:
train_model_reg(X, Y, skf, batch, max_epochs, criterion)

Treinando modelo ----- Fold 1
f1: 0.9717813051146384
Accuracy: 0.9722222222222222
Treinando modelo ----- Fold 2
f1: 0.9717813051146384
Accuracy: 0.9722222222222222
Treinando modelo ----- Fold 3
f1: 0.9709618874773139
Accuracy: 0.9722222222222222
Treinando modelo ----- Fold 4
f1: 0.974320987654321
Accuracy: 0.9714285714285714
Treinando modelo ----- Fold 5
f1: 1.0
Accuracy: 1.0

Resultado modelo ----- F1: 0.9777690970721824,  accuracy:0.9776190476190475
